# Hafta 3 — Doğrusal Regresyon

> Hücreleri **yukarıdan aşağıya sırayla** çalıştırın; sonraki hücreler önceki hücrelerde tanımlanan değişken ve fonksiyonları kullanır. Tek başına çalışmayan gösterim amaçlı kod parçaları markdown olarak bırakılmıştır. Veri setleri ilk çalıştırmada internetten indirilir.

In [ ]:
# Gerekli paketler kurulu değilse aşağıdaki satırın başındaki # işaretini kaldırıp çalıştırın
# %pip install numpy pandas matplotlib seaborn scikit-learn scipy

## Basit Anlatım: Bu Hafta Ne Öğreneceğiz?

Evlerin büyüklüğünü ve fiyatını bir grafiğe noktalar halinde yerleştirdiğinizi düşünün. Noktaların arasından en uygun şekilde geçen düz çizgiyi bulmaya doğrusal regresyon denir. Bu çizgi bulunduktan sonra yeni bir evin büyüklüğüne bakarak fiyatını tahmin etmek mümkün olur.

Bilgisayar bu çizgiyi tek seferde bulamaz. İşe rastgele bir çizgiyle başlar ve tahminlerinin ne kadar hatalı olduğunu hesaplar. Sonra çizgiyi hatayı azaltacak yönde biraz kaydırır. Bu işlem, hata yeterince küçülene kadar tekrarlanır. Sisli bir dağda yalnızca ayağınızın altındaki eğime bakarak adım adım vadiye inmek de aynı mantıkla işler.

---

## Bu Haftanın Hedefleri

- Doğrusal regresyonun ne öğrendiğini anlamak
- En iyi doğruyu kapalı form çözümle elle hesaplamak
- Gradyan iniş algoritmasını sıfırdan kodlamak
- MSE, MAE ve R² metriklerini yorumlamak

---

## 1. Model: Bir Doğru Uydurmak

Bir kafede günlük sıcaklık ile satılan soğuk içecek sayısını inceleyelim:

| Sıcaklık (x) | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| Satış (y) | 2 | 4 | 5 | 4 | 5 |

Aradığımız model bir doğrudur:

```
ŷ = w·x + b         w: eğim (x bir artınca ŷ ne kadar artar?)   b: kesişim
```

Birden çok öznitelik varsa model `ŷ = w₁x₁ + w₂x₂ + ... + b` biçimini alır.

**"En iyi" doğru hangisi?** Her nokta için hata `yᵢ − ŷᵢ` olsun. Hataların karelerinin ortalamasını (**MSE**) en küçük yapan doğru en iyisidir. Kare almanın nedeni, pozitif ve negatif hataların birbirini götürmesini önlemek ve büyük hataları daha çok cezalandırmaktır.

---

## 2. Kapalı Form Çözüm: Doğruyu Elle Bulmak

MSE'yi en küçük yapan w ve b şu formüllerle bulunur:

```
      Σ (xᵢ − x̄)(yᵢ − ȳ)
w = ──────────────────────          b = ȳ − w·x̄
         Σ (xᵢ − x̄)²
```

x̄ = 3, ȳ = 4:

| xᵢ | yᵢ | xᵢ − x̄ | yᵢ − ȳ | çarpım | (xᵢ − x̄)² |
|---|---|---|---|---|---|
| 1 | 2 | −2 | −2 | 4 | 4 |
| 2 | 4 | −1 | 0 | 0 | 1 |
| 3 | 5 | 0 | 1 | 0 | 0 |
| 4 | 4 | 1 | 0 | 0 | 1 |
| 5 | 5 | 2 | 1 | 2 | 4 |
| | | | **Toplam** | **6** | **10** |

w = 6 / 10 = **0.6**,  b = 4 − 0.6 × 3 = **2.2**  →  **ŷ = 0.6x + 2.2**

Yorum: Sıcaklık 1 °C artınca satış ortalama 0.6 adet artıyor.

Çok değişkenli durumda aynı çözüm matrislerle yazılır: `θ = (XᵀX)⁻¹Xᵀy` (normal denklem).

In [ ]:
import numpy as np

x = np.array([1, 2, 3, 4, 5], dtype=float)
y = np.array([2, 4, 5, 4, 5], dtype=float)

X = np.column_stack([np.ones_like(x), x])        # [1, x] sütunları
theta = np.linalg.solve(X.T @ X, X.T @ y)
print(theta)                                     # [2.2 0.6] → b = 2.2, w = 0.6

Öznitelik sayısı çok büyükse bu matris işlemi pahalılaşır. Bu durumda gradyan iniş kullanılır.

---

## 3. Gradyan İniş

### Çalışma mantığı

1. w ve b için rastgele (ör. 0) değerlerle başla.
2. Tahminleri ve hatayı hesapla.
3. Hatanın w ve b'ye göre eğimini (gradyanı) hesapla.
4. Parametreleri eğimin **tersi** yönünde küçük bir adım kaydır.
5. Hata yeterince küçülene kadar 2–4. adımları tekrarla.

```
w ← w − α · ∂L/∂w          ∂L/∂w = (2/n) Σ (ŷᵢ − yᵢ)·xᵢ
b ← b − α · ∂L/∂b          ∂L/∂b = (2/n) Σ (ŷᵢ − yᵢ)
```

α (**öğrenme oranı**) adım büyüklüğüdür. Tahminler büyük çıkarsa gradyan pozitif olur ve w, b küçültülür; tahminler küçük çıkarsa tersi olur.

### Sıfırdan kod

In [ ]:
import matplotlib.pyplot as plt

def gradyan_inis(x, y, alpha=0.05, epoch=2000):
    w, b, n = 0.0, 0.0, len(x)
    kayiplar = []
    for _ in range(epoch):
        hata = (w * x + b) - y                  # tahmin − gerçek
        kayiplar.append((hata ** 2).mean())     # MSE
        w -= alpha * (2 / n) * (hata * x).sum()
        b -= alpha * (2 / n) * hata.sum()
    return w, b, kayiplar

w, b, kayiplar = gradyan_inis(x, y)
print(f"w = {w:.3f}, b = {b:.3f}")              # w ≈ 0.600, b ≈ 2.200

plt.plot(kayiplar); plt.yscale("log")
plt.xlabel("Adım"); plt.ylabel("MSE"); plt.show()

Gradyan iniş, kapalı form çözümle aynı sonuca ulaştı.

### Öğrenme oranının etkisi

| α | Davranış |
|---|---|
| Çok küçük | Çok yavaş ilerler |
| Uygun | Hızlı ve düzgün biçimde minimuma iner |
| Çok büyük | Minimumun üzerinden atlar, hata **artar** |

Öznitelikler farklı ölçeklerdeyse gradyan iniş zigzag çizerek yavaş ilerler. Bu yüzden önce **ölçekleme** yapılır (Hafta 2).

---

## 4. Değerlendirme Metrikleri

Modelimiz ŷ = 0.6x + 2.2 için hatalar: −0.8, 0.6, 1.0, −0.6, −0.2

| Metrik | Hesap | Sonuç | Anlamı |
|---|---|---|---|
| **MSE** | Hataların karelerinin ortalaması | 2.4 / 5 = **0.48** | Büyük hataları sert cezalandırır |
| **RMSE** | √MSE | **0.69** | Hedefle aynı birimde ortalama hata |
| **MAE** | Mutlak hataların ortalaması | 3.2 / 5 = **0.64** | Aykırı değerlere daha dayanıklı |
| **R²** | 1 − (hata kareleri / toplam değişkenlik) | 1 − 2.4/6 = **0.60** | "Satışlardaki değişkenliğin %60'ı açıklanıyor" |

R² = 1 mükemmel tahmindir; R² = 0 ise model, her zaman ortalamayı tahmin etmekten daha iyi değildir.

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_tahmin = 0.6 * x + 2.2
print("MSE:", mean_squared_error(y, y_tahmin))     # 0.48
print("MAE:", mean_absolute_error(y, y_tahmin))    # 0.64
print("R² :", r2_score(y, y_tahmin))               # 0.60

---

## 5. Polinom Regresyon

İlişki eğriyse düz bir doğru yetersiz kalır. Çözüm, x², x³ gibi yeni öznitelikler eklemektir. Model parametrelerine göre hâlâ doğrusal olduğu için aynı yöntemler kullanılır.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

np.random.seed(0)
xp = np.sort(np.random.uniform(-3, 3, 30))
yp = 0.5 * xp**2 - xp + 2 + np.random.normal(0, 1, 30)    # gerçek ilişki: ikinci derece
cizim = np.linspace(-3, 3, 200).reshape(-1, 1)

plt.scatter(xp, yp, color="black")
for derece in [1, 2, 15]:
    m = make_pipeline(PolynomialFeatures(derece), LinearRegression()).fit(xp.reshape(-1, 1), yp)
    plt.plot(cizim, m.predict(cizim), label=f"Derece {derece}")
plt.ylim(-3, 15); plt.legend(); plt.show()

Derece 1 eğriyi yakalayamaz, derece 15 ise her noktaya uymaya çalışıp salınır. Bu iki sorunu (eksik ve aşırı öğrenme) Hafta 4'te inceleyeceğiz.

---

## 6. Laboratuvar: Kendi Gradyan İnişimiz ve scikit-learn

California konut verisinde (8 öznitelik) kendi çok değişkenli gradyan inişimizi scikit-learn ile karşılaştıralım:

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

veri = fetch_california_housing()
X_tr, X_te, y_tr, y_te = train_test_split(veri.data, veri.target, test_size=0.2, random_state=42)
sc = StandardScaler().fit(X_tr)
X_tr, X_te = sc.transform(X_tr), sc.transform(X_te)

# Çok değişkenli gradyan iniş (matris biçimi)
w, b = np.zeros(X_tr.shape[1]), 0.0
for _ in range(1000):
    hata = X_tr @ w + b - y_tr
    w -= 0.1 * (2 / len(y_tr)) * (X_tr.T @ hata)
    b -= 0.1 * (2 / len(y_tr)) * hata.sum()

sk = LinearRegression().fit(X_tr, y_tr)
print("Bizim R²  :", round(r2_score(y_te, X_te @ w + b), 4))
print("sklearn R²:", round(sk.score(X_te, y_te), 4))          # ikisi de ≈ 0.58

**Görevler:**
1. Ölçeklemeyi kaldırıp gradyan inişi tekrar çalıştırın. Ne oluyor?
2. α = 0.001, 0.01 ve 0.1 için kayıp eğrilerini aynı grafikte çizin.
3. Katsayılara bakarak ev fiyatını en çok etkileyen özniteliği bulun.

---

## 7. Özet

- Doğrusal regresyon, `ŷ = wx + b` doğrusunu MSE'yi en küçük yapacak şekilde öğrenir.
- **Kapalı form** tek adımda kesin çözüm verir; **gradyan iniş** adım adım yaklaşır ve büyük veride kullanılır.
- Öğrenme oranı çok küçükse eğitim yavaş, çok büyükse kararsız olur.
- MSE/RMSE/MAE hatanın büyüklüğünü, R² ise açıklanan değişkenliği ölçer.

## Kendinizi Sınayın

1. x = [0, 1, 2], y = [1, 3, 5] verisi için w ve b'yi elle hesaplayın.
2. Eğitim sırasında kayıp sürekli artıyorsa ilk neyi değiştirirsiniz?
3. Test verisinde R² = −0.3 çıkması ne anlama gelir?

## Çalışma Alanı

Alıştırmalar ve "Kendinizi Sınayın" soruları için aşağıdaki hücreleri kullanın.